# Clasificación de mejora del retraso — LightGBM

Predice si el retraso de un tren va a **mejorar** (reducirse) en los próximos 10 minutos.

- **Target**: `delta_delay_10m`
- **Clase 1** (mejora): `delta_delay_10m < 0`
- **Clase 0** (no mejora): `delta_delay_10m >= 0`

Validación temporal (dentro de enero 2025):
- Train → 70 % inicial
- Val → 15 % siguiente
- Test → 15 % final

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..')))

import io
import warnings
import gc
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve,
)
from sklearn.dummy import DummyClassifier

from src.common.minio_client import download_df_parquet

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

## Configuración

In [ ]:
ACCESS_KEY = os.environ["MINIO_ACCESS_KEY"]
SECRET_KEY = os.environ["MINIO_SECRET_KEY"]

YEAR          = 2025
MONTHS        = range(1, 2)
DATA_TEMPLATE = "grupo5/final/year={year}/month={month:02d}/dataset_final.parquet"

TARGET_DELTA = "delta_delay_10m"  # cambiar para otro horizonte: _20m, _30m, _45m, _60m, etc.
TARGET       = "target_mejora"

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
SEED        = 42

EXCLUDE_COLS = {
    "date", "match_key", "merge_time", "timestamp_start",
    "target_delay_10m", "target_delay_20m", "target_delay_30m",
    "target_delay_45m", "target_delay_60m", "target_delay_end",
    "delta_delay_10m", "delta_delay_20m", "delta_delay_30m",
    "delta_delay_45m", "delta_delay_60m", "delta_delay_end",
    "seconds_to_next_alert", "alert_in_next_15m", "alert_in_next_30m",
}

CAT_FEATURES = ["route_id", "direction", "category", "tipo_referente",
                "stop_id", "is_weekend", "is_unscheduled", "temp_extreme",
                "afecta_previo", "afecta_durante", "afecta_despues",
                "is_alert_just_published", "has_alert"]

LGBM_PARAMS = {
    "objective":         "binary",
    "metric":            "auc",
    "learning_rate":     0.05,
    "num_leaves":        127,
    "min_child_samples": 100,
    "feature_fraction":  0.8,
    "bagging_fraction":  0.8,
    "bagging_freq":      5,
    "reg_alpha":         0.1,
    "reg_lambda":        1.0,
    "n_jobs":            -1,
    "verbose":           -1,
    "seed":              SEED,
}
NUM_BOOST_ROUND = 3000
EARLY_STOPPING  = 50


## Helpers

In [ ]:
def load_months(months: range) -> pd.DataFrame:
    dfs = []
    for month in months:
        path = DATA_TEMPLATE.format(year=YEAR, month=month)
        try:
            df = download_df_parquet(ACCESS_KEY, SECRET_KEY, path)
            total = len(df)
            df = df.dropna(subset=[TARGET_DELTA])
            mb = df.memory_usage(deep=True).sum() / 1e6
            print(f"  ✓ month={month:02d}  {total:>10,} filas  →  {len(df):>10,} tras filtrado  ~{mb:.0f} MB")
            dfs.append(df)
        except Exception as e:
            print(f"  ✗ month={month:02d}  no encontrado ({e})")
    return pd.concat(dfs, ignore_index=True)


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Feature engineering: añade variables derivadas."""
    df = df.copy()

    # Tendencia e inercia del retraso
    df["delay_change"]      = df["delay_seconds"] - df["lagged_delay_1"]
    df["delay_change_prev"] = df["lagged_delay_1"] - df["lagged_delay_2"]
    df["delay_accel"]       = df["delay_change"] - df["delay_change_prev"]

    # Retraso relativo a la media de la ruta y de la estación
    df["delay_vs_route"]   = df["delay_seconds"] - df["route_rolling_delay"]
    df["delay_vs_station"] = df["delay_seconds"] - df["station_delay_10m"].fillna(df["delay_seconds"])
    df["station_trend"]    = df["station_delay_10m"] - df["station_delay_20m"]

    # Magnitud del retraso respecto al tiempo restante
    df["delay_time_ratio"] = df["delay_seconds"] / (df["scheduled_time_to_end"] + 1.0)

    # Impacto agregado de alertas
    df["has_alert"]    = (df["n_eventos_afectando"] > 0).astype(np.int8)
    df["alert_impact"] = df["afecta_previo"] + df["afecta_durante"] + df["afecta_despues"]

    return df


def encode_categoricals(df_train: pd.DataFrame, df_val: pd.DataFrame,
                        df_test: pd.DataFrame) -> tuple:
    for col in CAT_FEATURES:
        if col not in df_train.columns:
            continue
        vocab = {v: i for i, v in enumerate(df_train[col].astype(str).unique())}
        df_train[col] = df_train[col].astype(str).map(vocab).astype(int)
        df_val[col]   = df_val[col].astype(str).map(vocab).fillna(-1).astype(int)
        df_test[col]  = df_test[col].astype(str).map(vocab).fillna(-1).astype(int)
    return df_train, df_val, df_test


def get_features(df: pd.DataFrame) -> list[str]:
    return [c for c in df.columns if c not in EXCLUDE_COLS and c != TARGET]


def compute_metrics(y_true, y_prob, y_pred, prefix="") -> dict:
    return {
        f"{prefix}roc_auc":   round(roc_auc_score(y_true, y_prob), 4),
        f"{prefix}pr_auc":    round(average_precision_score(y_true, y_prob), 4),
        f"{prefix}accuracy":  round(accuracy_score(y_true, y_pred), 4),
        f"{prefix}precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        f"{prefix}recall":    round(recall_score(y_true, y_pred, zero_division=0), 4),
        f"{prefix}f1":        round(f1_score(y_true, y_pred, zero_division=0), 4),
    }


## 1. Carga de datos

In [ ]:
print(f"Cargando datos (meses {list(MONTHS)})...")
df = load_months(MONTHS)
print(f"  Total: {len(df):,} filas\n")
df.info(memory_usage="deep")

## 2. Inspección inicial

In [ ]:
# Nulos
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
if len(null_pct) > 0:
    print("Columnas con nulos:")
    for col, pct in null_pct.items():
        print(f"  {col:<35s} {pct:.2f}%")
else:
    print("Sin nulos.")

# Rango temporal
if not pd.api.types.is_datetime64_any_dtype(df["date"]):
    df["date"] = pd.to_datetime(df["date"])
for col in ["merge_time", "timestamp_start"]:
    if col in df.columns and not pd.api.types.is_datetime64_any_dtype(df[col]):
        df[col] = pd.to_datetime(df[col], errors="coerce")

print(f"\nRango date:       {df['date'].min()} → {df['date'].max()}")
print(f"Rango merge_time: {df['merge_time'].min()} → {df['merge_time'].max()}")

# Descriptivas del delta
df[TARGET_DELTA].describe()

## 3. Feature Engineering


In [ ]:
df = add_features(df)
print(f"Features añadidas. Shape: {df.shape}")


## 4. Definición del target binario


In [ ]:
# Definición de la lógica
df[TARGET] = (df[TARGET_DELTA] < 0).astype(np.int8)

# Distribución
counts = df[TARGET].value_counts().sort_index()
pcts   = df[TARGET].value_counts(normalize=True).sort_index() * 100
print(f"Distribución de '{TARGET}':")
print(f"  Clase 0 (no mejora): {counts[0]:>10,} ({pcts[0]:.1f}%)")
print(f"  Clase 1 (mejora):    {counts[1]:>10,} ({pcts[1]:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(["No mejora (0)", "Mejora (1)"], counts.values,
            color=["#e74c3c", "#2ecc71"], edgecolor="black")
axes[0].set_title("Distribución de clases")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + len(df)*0.005, f"{v:,}\n({pcts.values[i]:.1f}%)", ha="center")

axes[1].hist(df[TARGET_DELTA].clip(-300, 300), bins=100, color="#3498db", edgecolor="black", alpha=0.7)
axes[1].axvline(0, color="red", linestyle="--", linewidth=2, label="Umbral (0)")
axes[1].set_title(f"Distribución de {TARGET_DELTA}")
axes[1].set_xlabel("Segundos")
axes[1].legend()
plt.tight_layout()
plt.show()

## 4. Split temporal + codificación de categóricas

In [ ]:
# Ordenar por merge_time
SORT_COL = "merge_time" if df["merge_time"].notna().sum() > 0 else "date"
df = df.sort_values(SORT_COL).reset_index(drop=True)

n = len(df)
i_train = int(n * TRAIN_RATIO)
i_val   = int(n * (TRAIN_RATIO + VAL_RATIO))

df_train = df.iloc[:i_train].copy()
df_val   = df.iloc[i_train:i_val].copy()
df_test  = df.iloc[i_val:].copy()

# Codificar categóricas con vocabulario del train
df_train, df_val, df_test = encode_categoricals(df_train, df_val, df_test)

# Features
feats = get_features(df_train)
print(f"Features ({len(feats)}): {feats}\n")

X_train, y_train = df_train[feats], df_train[TARGET]
X_val,   y_val   = df_val[feats],   df_val[TARGET]
X_test,  y_test  = df_test[feats],  df_test[TARGET]

for name, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"  {name:>5s}: {len(X):>10,} filas  |  clase 1: {y.mean()*100:.1f}%")

# Liberar df grande
del df
gc.collect()

## 5. Baseline

In [ ]:
dummy = DummyClassifier(strategy="most_frequent", random_state=SEED)
dummy.fit(X_train, y_train)

for name, X, y in [("val", X_val, y_val), ("test", X_test, y_test)]:
    y_pred_d = dummy.predict(X)
    acc = accuracy_score(y, y_pred_d)
    f1  = f1_score(y, y_pred_d, average="binary", zero_division=0)
    print(f"Baseline {name}: accuracy={acc:.4f}  f1={f1:.4f}")

print("\n→ Este es el mínimo que el modelo debe superar.")

## 6. Entrenamiento LightGBM

In [ ]:
# Detectar desbalanceo
class_ratio = y_train.mean()
is_imbalanced = class_ratio < 0.3 or class_ratio > 0.7
LGBM_PARAMS["is_unbalance"] = is_imbalanced
print(f"Clase 1 en train: {class_ratio:.3f}  →  is_unbalance={is_imbalanced}")

# Datasets LightGBM
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val   = lgb.Dataset(X_val,   label=y_val, reference=lgb_train)

print(f"\nEntrenando LightGBM (target={TARGET_DELTA})...")
model = lgb.train(
    LGBM_PARAMS,
    lgb_train,
    num_boost_round=NUM_BOOST_ROUND,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(EARLY_STOPPING, verbose=False),
        lgb.log_evaluation(100),
    ],
)

print(f"\nMejor iteración: {model.best_iteration}")

## 7. Evaluación (umbral 0.5)

In [ ]:
# Predicciones
y_prob_train = model.predict(X_train, num_iteration=model.best_iteration)
y_prob_val   = model.predict(X_val,   num_iteration=model.best_iteration)
y_prob_test  = model.predict(X_test,  num_iteration=model.best_iteration)

y_pred_val  = (y_prob_val  >= 0.5).astype(int)
y_pred_test = (y_prob_test >= 0.5).astype(int)

# Métricas
m_val  = compute_metrics(y_val,  y_prob_val,  y_pred_val,  prefix="val_")
m_test = compute_metrics(y_test, y_prob_test, y_pred_test, prefix="test_")

print("Métricas val:");  [print(f"  {k}: {v}") for k, v in m_val.items()]
print("Métricas test:"); [print(f"  {k}: {v}") for k, v in m_test.items()]

# Classification report
print("\n── Classification Report (test) ──")
print(classification_report(y_test, y_pred_test, target_names=["No mejora", "Mejora"]))

In [ ]:
# Matriz de confusión
for name, y_true, y_pred in [("Val", y_val, y_pred_val), ("Test", y_test, y_pred_test)]:
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues",
                xticklabels=["No mejora", "Mejora"],
                yticklabels=["No mejora", "Mejora"], ax=ax)
    ax.set_xlabel("Predicción"); ax.set_ylabel("Real")
    ax.set_title(f"Confusión — {name}")
    plt.tight_layout(); plt.show()

In [ ]:
# Curvas ROC y PR
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fpr, tpr, _ = roc_curve(y_test, y_prob_test)
axes[0].plot(fpr, tpr, linewidth=2, label=f"AUC={m_test['test_roc_auc']:.3f}")
axes[0].plot([0,1],[0,1], "k--", linewidth=1)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC — Test"); axes[0].legend()

prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_prob_test)
axes[1].plot(rec_curve, prec_curve, linewidth=2, label=f"PR-AUC={m_test['test_pr_auc']:.3f}")
axes[1].axhline(y_test.mean(), color="gray", linestyle="--", label=f"Baseline={y_test.mean():.3f}")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall — Test"); axes[1].legend()

plt.tight_layout(); plt.show()

## 8. Optimización de umbral

In [ ]:
thresholds = np.arange(0.20, 0.80, 0.01)
f1s, precs, recs = [], [], []

for t in thresholds:
    yp = (y_prob_val >= t).astype(int)
    f1s.append(f1_score(y_val, yp, zero_division=0))
    precs.append(precision_score(y_val, yp, zero_division=0))
    recs.append(recall_score(y_val, yp, zero_division=0))

best_idx = np.argmax(f1s)
BEST_THRESHOLD = thresholds[best_idx]

print(f"Mejor umbral (val): {BEST_THRESHOLD:.2f}  →  F1={f1s[best_idx]:.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1s, label="F1", linewidth=2)
ax.plot(thresholds, precs, label="Precision", linestyle="--")
ax.plot(thresholds, recs, label="Recall", linestyle="--")
ax.axvline(BEST_THRESHOLD, color="black", linestyle=":", label=f"Óptimo={BEST_THRESHOLD:.2f}")
ax.axvline(0.5, color="gray", linestyle=":", alpha=0.5, label="Default=0.5")
ax.set_xlabel("Umbral"); ax.set_ylabel("Métrica")
ax.set_title("Métricas vs Umbral (val)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Evaluación con umbral óptimo en test
y_pred_test_opt = (y_prob_test >= BEST_THRESHOLD).astype(int)
m_test_opt = compute_metrics(y_test, y_prob_test, y_pred_test_opt, prefix="test_opt_")

print(f"Métricas test (umbral={BEST_THRESHOLD:.2f}):")
[print(f"  {k}: {v}") for k, v in m_test_opt.items()]

print(f"\n── Classification Report (test, umbral={BEST_THRESHOLD:.2f}) ──")
print(classification_report(y_test, y_pred_test_opt, target_names=["No mejora", "Mejora"]))

# Comparativa
print("\n── Comparativa ──")
print(f"{'':20s} {'Baseline':>10s} {'LGBM 0.5':>10s} {'LGBM ópt':>10s}")
dummy_acc = accuracy_score(y_test, dummy.predict(X_test))
print(f"{'Accuracy':<20s} {dummy_acc:>10.4f} {m_test['test_accuracy']:>10.4f} {m_test_opt['test_opt_accuracy']:>10.4f}")
print(f"{'F1':<20s} {'—':>10s} {m_test['test_f1']:>10.4f} {m_test_opt['test_opt_f1']:>10.4f}")
print(f"{'ROC-AUC':<20s} {'0.5':>10s} {m_test['test_roc_auc']:>10.4f} {m_test_opt['test_opt_roc_auc']:>10.4f}")

## 9. Feature importance

In [ ]:
importance = pd.DataFrame({
    "feature":    model.feature_name(),
    "importance":  model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False)

importance["pct"] = (importance["importance"] / importance["importance"].sum() * 100).round(2)

print(f"Top 20 features:")
print(importance.head(20).to_string(index=False))

# Gráfico top 20
top = importance.head(20)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(len(top)), top["pct"].values[::-1], color="#3498db", edgecolor="black")
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top["feature"].values[::-1])
ax.set_xlabel("Importancia (% gain)")
ax.set_title("Top 20 Features")
plt.tight_layout(); plt.show()

## 10. Análisis de errores

In [ ]:
df_err = df_test.copy()
df_err["y_true"] = y_test.values
df_err["y_pred"] = y_pred_test_opt
df_err["y_prob"] = y_prob_test

df_err["error_type"] = "Correcto"
df_err.loc[(df_err["y_true"] == 0) & (df_err["y_pred"] == 1), "error_type"] = "FP"
df_err.loc[(df_err["y_true"] == 1) & (df_err["y_pred"] == 0), "error_type"] = "FN"

print("Distribución de predicciones:")
print(df_err["error_type"].value_counts().to_string())

In [ ]:
# Ejemplos de FP y FN
show_cols = ["route_id", "stop_id", "delay_seconds", "y_prob", TARGET_DELTA]
show_cols = [c for c in show_cols if c in df_err.columns]

print("── Falsos Positivos (predice mejora, no mejora) ──")
print(df_err[df_err["error_type"] == "FP"][show_cols].head(10).to_string(index=False))

print("\n── Falsos Negativos (predice no mejora, sí mejora) ──")
print(df_err[df_err["error_type"] == "FN"][show_cols].head(10).to_string(index=False))

In [ ]:
# Métricas por route_id
route_metrics = []
for route, g in df_err.groupby("route_id"):
    if len(g) < 50:
        continue
    yt, yp, ypr = g["y_true"], g["y_pred"], g["y_prob"]
    try:
        auc = roc_auc_score(yt, ypr) if yt.nunique() > 1 else np.nan
    except:
        auc = np.nan
    route_metrics.append({
        "route_id": route, "n": len(g),
        "accuracy": round(accuracy_score(yt, yp), 4),
        "f1": round(f1_score(yt, yp, zero_division=0), 4),
        "auc": round(auc, 4) if not np.isnan(auc) else np.nan,
    })

route_df = pd.DataFrame(route_metrics).sort_values("f1")
print("Top 10 PEORES rutas por F1:")
print(route_df.head(10).to_string(index=False))
print("\nTop 10 MEJORES rutas por F1:")
print(route_df.tail(10).to_string(index=False))

In [ ]:
# Métricas por hora
if "hour_sin" in df_err.columns and "hour_cos" in df_err.columns:
    df_err["hour"] = (
        np.arctan2(df_err["hour_sin"], df_err["hour_cos"]) / (2*np.pi) * 24
    ).round().astype(int) % 24

    hour_m = []
    for h, g in df_err.groupby("hour"):
        yt, yp = g["y_true"], g["y_pred"]
        hour_m.append({"hour": h, "n": len(g),
                       "f1": f1_score(yt, yp, zero_division=0),
                       "error_rate": (yt != yp).mean()})

    hour_df = pd.DataFrame(hour_m).sort_values("hour")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(hour_df["hour"], hour_df["f1"], color="#2ecc71", edgecolor="black")
    axes[0].set_xlabel("Hora"); axes[0].set_ylabel("F1")
    axes[0].set_title("F1 por hora"); axes[0].set_xticks(range(0, 24))

    axes[1].bar(hour_df["hour"], hour_df["error_rate"]*100, color="#e74c3c", edgecolor="black")
    axes[1].set_xlabel("Hora"); axes[1].set_ylabel("Error %")
    axes[1].set_title("Tasa de error por hora"); axes[1].set_xticks(range(0, 24))
    plt.tight_layout(); plt.show()

# Laborable vs fin de semana
if "is_weekend" in df_err.columns:
    print("Métricas por tipo de día:")
    for wk, label in [(0, "Laborable"), (1, "Fin de semana")]:
        g = df_err[df_err["is_weekend"] == wk]
        if len(g) == 0: continue
        yt, yp = g["y_true"], g["y_pred"]
        print(f"  {label:<15s}: acc={accuracy_score(yt,yp):.4f}  f1={f1_score(yt,yp,zero_division=0):.4f}  n={len(g):,}")

In [ ]:
# Top estaciones por tasa de error
if "stop_id" in df_err.columns:
    stop_m = []
    for stop, g in df_err.groupby("stop_id"):
        if len(g) < 100: continue
        yt, yp = g["y_true"], g["y_pred"]
        stop_m.append({"stop_id": stop, "n": len(g),
                       "error_rate": round((yt != yp).mean(), 4),
                       "f1": round(f1_score(yt, yp, zero_division=0), 4)})
    stop_df = pd.DataFrame(stop_m).sort_values("error_rate", ascending=False)
    print("Top 15 estaciones con MAYOR error:")
    print(stop_df.head(15).to_string(index=False))
    print("\nTop 15 estaciones con MENOR error:")
    print(stop_df.tail(15).to_string(index=False))

## 11. Resumen estadístico del modelo

In [ ]:
# Resumen global de rendimiento en TEST
metric_names = ["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]

def strip_prefix(metrics_dict, prefix):
    out = {}
    for k, v in metrics_dict.items():
        if k.startswith(prefix):
            out[k[len(prefix):]] = v
    return out

m_test_base = strip_prefix(m_test, "test_")
m_test_best = strip_prefix(m_test_opt, "test_opt_")

summary = pd.DataFrame(
    [
        {"Escenario": "Umbral 0.50", **m_test_base},
        {"Escenario": f"Umbral óptimo ({BEST_THRESHOLD:.2f})", **m_test_best},
    ]
).set_index("Escenario")

print("=== Resumen principal de métricas (TEST) ===")
display(summary[metric_names].round(4))

# Comparativa de mejora al pasar del umbral 0.50 al óptimo
opt_label = f"Umbral óptimo ({BEST_THRESHOLD:.2f})"
delta = (summary.loc[opt_label] - summary.loc["Umbral 0.50"]).to_frame("Delta")
print("\n=== Cambio de métricas con umbral óptimo ===")
display(delta.round(4))

# Estadísticas adicionales de clasificación con umbral óptimo
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test_opt).ravel()
total = tn + fp + fn + tp

spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
rec = recall_score(y_test, y_pred_test_opt, zero_division=0)
bal_acc = (rec + spec) / 2 if not np.isnan(spec) else np.nan

stats_extra = pd.DataFrame(
    {
        "Métrica": [
            "Total muestras",
            "Positivos reales",
            "Negativos reales",
            "TP",
            "FP",
            "FN",
            "TN",
            "Tasa falsos positivos (FPR)",
            "Tasa falsos negativos (FNR)",
            "Balanced accuracy",
        ],
        "Valor": [
            int(total),
            int((y_test == 1).sum()),
            int((y_test == 0).sum()),
            int(tp),
            int(fp),
            int(fn),
            int(tn),
            round(fp / (fp + tn), 4) if (fp + tn) > 0 else np.nan,
            round(fn / (fn + tp), 4) if (fn + tp) > 0 else np.nan,
            round(bal_acc, 4) if not np.isnan(bal_acc) else np.nan,
        ],
    }
)

print("\n=== Estadísticas adicionales (TEST, umbral óptimo) ===")
display(stats_extra)

## Siguientes pruebas

1. Probar otros horizontes: `delta_delay_20m`, `delta_delay_30m`, etc. (cambiar `TARGET_DELTA`)
2. Entrenar un modelo por horizonte y comparar métricas
3. Calibración de probabilidades (`CalibratedClassifierCV`)
4. Validación walk-forward (expanding window)
5. Más features históricas: más lags, medias móviles por estación, tendencias
6. Comparar con CatBoost / XGBoost
7. Subir resultados a WandB para tracking